In [1]:
# Import all necessary libraries
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time

In [2]:
# Check device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define hyper-parameters
input_size = 784
num_classes = 10
num_epochs = 20
learning_rate = 0.01
batch_size = 100

# MNIST Dataset
train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, transform=transforms.ToTensor()
)

# Data loader
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset, batch_size=batch_size, shuffle=False
)

In [3]:
# Network #1: 4-layer network
class Network1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 20),
            nn.ReLU(),
            nn.Linear(20, 50),
            nn.ReLU(),
            nn.Linear(50, 20),
            nn.ReLU(),
            nn.Linear(20, num_classes),
        )

    def forward(self, x):
        return self.layers(x.view(-1, input_size))


# Network #2: 6-layer network
class Network2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, num_classes),
        )

    def forward(self, x):
        return self.layers(x.view(-1, input_size))


# Network #3: 6-layer network
class Network3(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Linear(10, 40),
            nn.ReLU(),
            nn.Linear(40, 70),
            nn.ReLU(),
            nn.Linear(70, 40),
            nn.ReLU(),
            nn.Linear(40, 10),
            nn.ReLU(),
            nn.Linear(10, num_classes),
        )

    def forward(self, x):
        return self.layers(x.view(-1, input_size))

In [4]:
def run_experiment(model_class, name):
    print(f"\nStarting: {name}")
    model = model_class().to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Train the model
    start = time.perf_counter()
    total_step = len(train_loader)
    for epoch in range(num_epochs):
        for i, (images, labels) in enumerate(train_loader):
            # Move tensors to the configured device
            images = images.reshape(-1, 28 * 28).to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backprpagation and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i + 1) % 100 == 0:
                print(
                    f"Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{total_step}], Loss: {loss.item():.4f}"
                )
                
    duration = time.perf_counter() - start
    print(f"{name} took {duration:.2f} s to train")

    # Test the model
    # In the test phase, don't need to compute gradients (for memory efficiency)
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            # images, labels = images.to(device), labels.to(device)
            images = images.reshape(-1, 28 * 28).to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Accuracy of the network on the 10,000 test images: {accuracy:.4f}")


experiments = [(Network1, "Model #1"), (Network2, "Model #2"), (Network3, "Model #3")]
for m_class, m_name in experiments:
    run_experiment(m_class, m_name)



Starting: Model #1
Epoch [1/20], Step [100/600], Loss: 0.4900
Epoch [1/20], Step [200/600], Loss: 0.3216
Epoch [1/20], Step [300/600], Loss: 0.3178
Epoch [1/20], Step [400/600], Loss: 0.3136
Epoch [1/20], Step [500/600], Loss: 0.1384
Epoch [1/20], Step [600/600], Loss: 0.2578
Epoch [2/20], Step [100/600], Loss: 0.0990
Epoch [2/20], Step [200/600], Loss: 0.0866
Epoch [2/20], Step [300/600], Loss: 0.2225
Epoch [2/20], Step [400/600], Loss: 0.3579
Epoch [2/20], Step [500/600], Loss: 0.1277
Epoch [2/20], Step [600/600], Loss: 0.1285
Epoch [3/20], Step [100/600], Loss: 0.2069
Epoch [3/20], Step [200/600], Loss: 0.1531
Epoch [3/20], Step [300/600], Loss: 0.1164
Epoch [3/20], Step [400/600], Loss: 0.2606
Epoch [3/20], Step [500/600], Loss: 0.3040
Epoch [3/20], Step [600/600], Loss: 0.1216
Epoch [4/20], Step [100/600], Loss: 0.1009
Epoch [4/20], Step [200/600], Loss: 0.2223
Epoch [4/20], Step [300/600], Loss: 0.2553
Epoch [4/20], Step [400/600], Loss: 0.1268
Epoch [4/20], Step [500/600], Loss

#### Which of the three models had the least amount of error for validation?

- At 95%, compared to 92 and 94, the model #1 had the least amount of error for validation.

#### How long it took to train each model?

- It took 81.99 s to train model #1, took 83.29 s to train model #2, and 86 s to train model #3.